**DIP Group Project**

**Members** :
1. Gubba Rohan (CB.SC.U4CSE24016)
2. Nikhil Jones A (CB.SC.U4CSE24031)
3. SH Nihil Mukkesh (CB.SC.U4CSE24531)
4. P.N.V Siddartha (CB.SC.U4CSE24737)

**Roadmap & Execution**

1. Applied each preprocessing technique individually on different image categories to understand its effect.
2. Compared the performance of techniques such as Gamma Correction, Contrast Stretching, Mean Filter, Median Filter, Minimum Filter, Laplacian, High Boost, Sobel Edge Detection, Bit Plane Slicing, Intensity Level Slicing, and Frequency Domain Filters.
3. Recorded the most effective technique(s) for each image category based on visual enhancement.
4. Observed that no single preprocessing technique produced the best result for every image type.
5. Experimented with different combinations and sequences of preprocessing operations.
6. Designed and evaluated multiple preprocessing pipelines using combinations of enhancement, sharpening, and feature extraction techniques.


# **Explanation**

**Stage 1: Applying each preprocessing on different image categories**

Observations from stage 1:

Through observation, we found out which techniques work best.



  _Format: Image (Description) → Best-performing technique(s)_
- writing (faded handwritten text) - l1, contrast stretch
- writing1 (faded handwritten text) - sobel, contrast stretch
- Fingerprint2 (fingerprint) - bit-plane 5, sobel
- tire1 (tire tracks) - minimum_filter
- underbed (person hiding under a bed) - gamma
- 2army (people in camouflage) - laplacian l1
- cctv2 (CCTV surveillance scene) - sobel

**Stage 2: Identifying pipeline for images with overexposed lighting**

- Gamma (y=1.8) -> Contrast Stretching -> High Boost (A = 1.5)
  - Used Gamma for reducing intensity of brighter areas and making the dark parts relatively more visible.
  - Used Contrast Stretching further to reduce the over-exposure in images.

**Stage 3: Identifying common pipeline**

- Observations:
  - We noticed that the techniques Sobel and Laplacian have been the most helpful, and used in common for all categories.
  - Bit Plane Slicing is good for processing images of fingerprints.
  - Contrast Stretching helps make handwritten/faded/erased texts easier to read.
- Template used to identify pipelines
  - Image -> GrayScale Conversion -> Noise Reduction -> Constrast Enhancement -> Image Sharpening -> Sobel/Bitplane Slicing -> Output
- Identified Pipelines:
  1. Input -> Grayscale Conv -> Meaner Filter -> Constrast Stretching -> Laplacian L1 Add -> Sobel
  2. Input -> Grayscale Conv -> Meaner Filter -> Constrast Stretching -> Laplacian L1 Add -> Bit Plane 5
  3. Input -> Grayscale Conv -> Meaner Filter -> Constrast Stretching -> Laplacian L1 Add -> Bit Plane 6

- Analysis of the Last Step:
  - Sobel helps in most cases to identify outlines of objects, people, buildings etc.
  - Bitplane slicing helps to modify images to make extracting fingerprints easier.


# **Justification**

**Pipeline 1**:
- We can use sobel to identify and isolate/focus on the subjects in the images.
- Subjects can be people, building, weapons, etc.
  - By emphasizing edges while suppressing relatively uniform regions, it makes the outlines of buildings, vehicles, persons, doors, windows, and other objects more distinct. This assists investigators in scene analysis, object localization, and evidence interpretation, especially when CCTV footage has low contrast or poor visibility.

**Pipeline 2 & 3**:
- In our observation, bitplane slicing highlights fingerprints by remove unnecessary skin in between. This helps systems scan fingerprints with higher accuracy

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [11]:
image_files = [
    "input/writing.png",
    "input/writing1.png",
    "input/fingerprint2.png",
    "input/tire1.png",
    "input/underbed.webp",
    "input/2army.jpg",
    "input/cctv2.png",
    "input/cctv5.png",
    "input/building1.jpg",
    "input/building2.jpg"
]

OUTPUT_FOLDER = "output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [12]:
mean_mask = np.ones((3,3), np.float32) / 9

L1 = np.array([
    [0,-1,0],
    [-1,4,-1],
    [0,-1,0]
], dtype=np.float32)

In [13]:
for file in image_files:

    img = cv2.imread(file)

    if img is None:
        print(f"Skipping {file}")
        continue


    # Grayscale


    gray = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2GRAY
    )

    # Mean Filter

    mean = cv2.filter2D(
        gray,
        -1,
        mean_mask
    )

    # Contrast Stretching

    contrast = cv2.normalize(
        mean,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    # Laplacian L1 (Addition)

    lap = cv2.filter2D(
        contrast,
        cv2.CV_32F,
        L1
    )

    lap_add = contrast.astype(np.float32) + lap

    lap_add = np.clip(
        lap_add,
        0,
        255
    ).astype(np.uint8)

    # PIPELINE 1
    # Sobel

    gx = cv2.Sobel(
        lap_add,
        cv2.CV_64F,
        1,
        0,
        ksize=3
    )

    gy = cv2.Sobel(
        lap_add,
        cv2.CV_64F,
        0,
        1,
        ksize=3
    )

    pipeline1 = np.sqrt(
        gx**2 + gy**2
    )

    pipeline1 = np.clip(
        pipeline1,
        0,
        255
    ).astype(np.uint8)

    # PIPELINE 2
    # Bit Plane 5

    pipeline2 = ((lap_add >> 5) & 1) * 255
    pipeline2 = pipeline2.astype(np.uint8)

    # PIPELINE 3
    # Bit Plane 6

    pipeline3 = ((lap_add >> 6) & 1) * 255
    pipeline3 = pipeline3.astype(np.uint8)

    # SAVE COMPARISON

    plt.figure(figsize=(16,4))

    titles = [
        "Original",
        "Pipeline 1\nSobel",
        "Pipeline 2\nBit Plane 5",
        "Pipeline 3\nBit Plane 6"
    ]

    images = [
        gray,
        pipeline1,
        pipeline2,
        pipeline3
    ]

    for i in range(4):
        plt.subplot(1,4,i+1)
        plt.imshow(images[i], cmap="gray")
        plt.title(titles[i])
        plt.axis("off")

    plt.tight_layout()

    name = os.path.splitext(
        os.path.basename(file)
    )[0]

    plt.savefig(
        os.path.join(
            OUTPUT_FOLDER,
            f"{name}_comparison.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"Processed: {file}")


Processed: input/writing.png
Processed: input/writing1.png
Processed: input/fingerprint2.png
Processed: input/tire1.png
Processed: input/underbed.webp
Processed: input/2army.jpg
Processed: input/cctv2.png
Processed: input/cctv5.png
Processed: input/building1.jpg
Processed: input/building2.jpg
